# SBND RWM Timing Calibration: Segments (Run 1 + Run 2)

- Per run: σ and σR − σL per chunk coloured by segment (median ± 3σ band), stacked bunch per segment with an asym Gaussian fit

## 1. Setup

- Inputs: `NTUPLE` = ntuple per run (`$SBND_RWM_NTUPLE_RUN1/2`), `CALIB_DIR` = calibration notebook output folder (`$SBND_RWM_OUT`), `DAQ_RUN2` = DAQ Run 2 run log (`$SBND_RWM_DAQ_RUN2`)
- Segments: `SEGMENTS` table in `lib/event_timestamps_run{1,2}.py`

In [ ]:
import os
import time
os.environ['TZ'] = 'UTC'
time.tzset()

import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcdefaults()
from matplotlib.ticker import FormatStrFormatter
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.legend_handler import HandlerTuple

import sys
sys.path.insert(0, '../lib')
import paths
from run_table import load_daq_runs
import event_timestamps_run1 as ev1
import event_timestamps_run2 as ev2
from calib_fit_helpers import load_run_raw
from registry import assign_segments
from stacking import apply_to_tree, load_calibration, stacked_bunch_fit
from plot_helpers import SEG_COLORS, draw_asym_fit, plot_chunk_series, time_axis

NTUPLE = {1: os.environ.get('SBND_RWM_NTUPLE_RUN1', '/path/to/run1_timing_calib_ntuple.root'),
          2: os.environ.get('SBND_RWM_NTUPLE_RUN2', '/path/to/run2_timing_calib_ntuple.root')}
CALIB_DIR = os.environ.get('SBND_RWM_OUT', '../output')   # calibration notebook output: CALIB_DIR/run{1,2}/sbnd_fine_calibration.csv
paths.set_paths(NTUPLE, CALIB_DIR)
DAQ_RUN2 = os.environ.get('SBND_RWM_DAQ_RUN2', '/path/to/DAQ Runs - DAQ Runs (Run 2).csv')   # DAQ run log (segment A span)
FIG_DIR = f'{CALIB_DIR}/figures'
os.makedirs(FIG_DIR, exist_ok=True)

FULL = 6.3   # figure width [in]
plt.rcParams.update({'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 7, 'legend.fontsize': 6, 'xtick.labelsize': 7,
                     'ytick.labelsize': 7, 'legend.title_fontsize': 6})

BUNCH_FIT_WINDOW = 2.5   # stacked-bunch fit window [σ]
RUNS = {
    'Run 1': dict(run=1, ev=ev1, seg_ylim=(1.45, 2.65),
                  seg_labels={'before_a': 'Before Rotation A', 'before_b': 'Before Rotation B (Unstable)',
                              'before_c': 'Before Rotation C', 'during': 'Rotation', 'after': 'After Rotation'}),
    'Run 2': dict(run=2, ev=ev2, seg_ylim=(1.25, 2.1),
                  seg_labels={'wide': 'Wide', 'normal_a': 'Normal A', 'normal_b': 'Normal B', 'narrow': 'Narrow'}),
}
VALUES = {'sigma': r'$\sigma$ [ns]', 'rl_asym': r'$\sigma_R - \sigma_L$ [ns]'}

## 2. Load Data

In [ ]:
for r in RUNS.values():
    r['df'] = assign_segments(load_calibration(f"{CALIB_DIR}/run{r['run']}/sbnd_fine_calibration.csv"), r['ev'])
    print(f"Run {r['run']}: {len(r['df'])} fitted chunks, segments {r['df']['seg'].value_counts().to_dict()}")
daq2 = load_daq_runs(DAQ_RUN2)
seg_a = daq2[daq2['run'].between(*ev2.SEGMENT_A_RUNS)]
SEG_A = (seg_a['start'].min(), seg_a['end'].max())

## 3. Segment Figures

- Slices stacked on their nearest bucket centre (core + edge buckets) inside the spill window (`calib_time_ns`, as in `apply_calibration.ipynb`); fit within ±`BUNCH_FIT_WINDOW` σ

In [ ]:
def plot_bunch(ax, r, seg):
    d = r['df']
    ch = d.index[d['seg'] == seg]
    P = d.loc[ch, 'period'].median()
    res = r['app'].loc[r['app']['calib_chunk'].isin(ch) & (r['app']['calib_status'] == 'in_spill'), 'calib_resid'].values
    f = stacked_bunch_fit(res, P, BUNCH_FIT_WINDOW)
    mid, n, mask, bw = f['mid'], f['n'], f['mask'], f['bw']
    ax.step(mid, n, where='mid', color='k', lw=0.7)
    ax.axvspan(mid[mask][0], mid[mask][-1], color='tab:grey', alpha=0.1)
    draw_asym_fit(ax, f, lambda side, s, e: rf'$\sigma_{side}$ {s:.3f} $\pm$ {e:.3f} ns')
    for line in ax.get_lines():
        line.set_linewidth(min(line.get_linewidth(), 1.0))
    ax.plot([], [], ' ', label=rf"$\chi^2$/ndof {f['chi_a']:.1f}")
    ax.set_ylim(0, 1.6 * n.max())
    ax.set_title(r['seg_labels'][seg], color=SEG_COLORS[seg], fontweight='bold', pad=3)
    ax.set_xlabel('OpFlashT0Corrected [ns]')
    ax.set_ylabel(f'Entries / [{bw:.2f} ns]')
    h, l = ax.get_legend_handles_labels()
    ax.legend([h[i] for i in (0, 2, 1)], [l[i] for i in (0, 2, 1)], ncol=2, loc='upper center', columnspacing=0.5,
              handlelength=1.0, handletextpad=0.4, borderaxespad=0.3)


def plot_run_segments(name):
    r = RUNS[name]
    d = r['df']
    raw = load_run_raw(r['run'])
    r['app'] = apply_to_tree(raw, d)
    del raw
    order = [s for s in r['seg_labels'] if s not in r['ev'].UNSTABLE_SEGMENTS]
    fig = plt.figure(figsize=(FULL, 8.9))
    gs = fig.add_gridspec(2, 1, height_ratios=(1, 1.15), hspace=0.42)
    top = gs[0].subgridspec(2, 1, hspace=0.12)
    bot = gs[1].subgridspec(2, 2, hspace=0.5, wspace=0.38)
    ax_s = fig.add_subplot(top[0])
    ax_a = fig.add_subplot(top[1], sharex=ax_s)
    x0 = SEG_A[0] if r['run'] == 2 else d['start'].min()   # Run 2: axis from segment A (not processed, red)
    for seg in order:
        g = d[d['seg'] == seg]
        for ax, col in ((ax_s, 'sigma'), (ax_a, 'rl_asym')):
            plot_chunk_series(ax, g, col, color=SEG_COLORS[seg], center=False)
    for seg in order:   # median ± 3σ band, σ = (q84 − q16) / 2 of the segment's chunk values
        g = d[d['seg'] == seg]
        a, b = g['start'].min(), g['end'].max()
        for ax, col in ((ax_s, 'sigma'), (ax_a, 'rl_asym')):
            q16, med, q84 = g[col].quantile([0.16, 0.5, 0.84])
            sig = (q84 - q16) / 2
            ax.fill_between([a, b], med - 3 * sig, med + 3 * sig, color=SEG_COLORS[seg], alpha=0.07, lw=0, zorder=1)
            ax.plot([a, b], [med, med], color=SEG_COLORS[seg], ls=':', lw=1.2, zorder=5, label=f'{med:.2f} ± {3 * sig:.2f} ns')
    u = d[d['unsegmented'] | d['seg'].isin(r['ev'].UNSTABLE_SEGMENTS)]
    for ax, col in ((ax_s, 'sigma'), (ax_a, 'rl_asym')):
        if len(u):
            plot_chunk_series(ax, u, col, color='k', center=False)
        for line in ax.get_lines():
            if line.get_linestyle() == '-':
                line.set_linewidth(0.8)
        r['ev'].plot_extended_downtime(ax)
        if r['run'] == 2:
            ax.axvspan(*SEG_A, color='tab:red', alpha=0.1, lw=0)
        time_axis(ax, (x0, d['end'].max()))
    ax_s.tick_params(labelbottom=False)
    if r['run'] == 2:
        ax_s.text(SEG_A[0] + (SEG_A[1] - SEG_A[0]) / 2, 0.5, 'Run 2 Segment A\n(not processed)', transform=ax_s.get_xaxis_transform(),
                  ha='center', va='center', color='tab:red', fontsize=7)
    ax_s.set_ylim(*r['seg_ylim'])
    ax_s.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    ax_s.set_ylabel(VALUES['sigma'])
    ax_a.axhline(0, color='k', ls=':', lw=0.8)
    ax_a.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    ax_a.set_ylabel(VALUES['rl_asym'])
    # one legend below both rows: segment, median ± 3σ of σ and σR − σL
    meds = {ax: ax.get_legend_handles_labels()[1] for ax in (ax_s, ax_a)}
    hs = [(Patch(color=SEG_COLORS[seg], alpha=0.15, lw=0), Line2D([], [], color=SEG_COLORS[seg], ls=':', lw=1.2)) for seg in order]
    ls = [f"{r['seg_labels'][seg]}\n$\\sigma$ = {ms}\n$\\sigma_R - \\sigma_L$ = {ma}"
          for seg, ms, ma in zip(order, meds[ax_s], meds[ax_a])]
    ax_a.legend(hs, ls, handler_map={tuple: HandlerTuple(ndivide=1)}, loc='upper center', bbox_to_anchor=(0.5, -0.28),
                ncol=len(order), title='Segment Median ± 3σ', handlelength=1.2, handletextpad=0.4, columnspacing=1.0,
                frameon=True, fancybox=False, edgecolor='0.6', framealpha=1)
    fig.align_ylabels([ax_s, ax_a])
    for j, seg in enumerate(order):
        plot_bunch(fig.add_subplot(bot[j // 2, j % 2]), r, seg)
    fig.savefig(f"{FIG_DIR}/calib_results_segments_run{r['run']}.png", dpi=300, bbox_inches='tight')
    plt.show()
    del r['app']

### Run 1

In [ ]:
plot_run_segments('Run 1')

### Run 2

In [ ]:
plot_run_segments('Run 2')

## 4. Segment Summary

- Per segment: chunks, duration (first to last entry), median ± 3σ of the chunk σ and σR − σL

In [ ]:
for name, r in RUNS.items():
    d = r['df']
    for seg in r['seg_labels']:
        g = d[d['seg'] == seg]
        vals = []
        for col in ('sigma', 'rl_asym'):
            q16, med, q84 = g[col].quantile([0.16, 0.5, 0.84])
            vals.append(f'{med:.2f} ± {3 * (q84 - q16) / 2:.2f}')
        span = (g['end'].max() - g['start'].min()).total_seconds() / 86400   # first to last entry of the segment
        print(f"{name} {seg:9s} chunks {len(g):3d}  duration {span:5.1f} d  σ {vals[0]} ns  σR − σL {vals[1]} ns")